# Stage 1.5 — Sanity-check the combined embedding

Loads `data/game_embeddings_matrix.npy` + `data/game_embeddings_index.pkl`
(produced by `build_combined_embeddings.py`), wraps them into the same
`{name: vector}` shape that `find_closest_games` expects, and prints the
top-10 cosine neighbors for a handful of pre-filled query games covering
distinct genres.

**This is a human gate before Stage 2** — eyeball the results; only move
on once they look reasonable. The last code cell is left empty for ad-hoc
queries.


In [1]:
import pickle
import numpy as np
import pandas as pd

# Load the combined embedding matrix + the index DataFrame.
E = np.load("../../data/game_embeddings_matrix.npy")
index_df = pd.read_pickle("../../data/game_embeddings_index.pkl")

# Wrap as {name: vector} so the existing find_closest_games (from
# test_embeddings.ipynb) works unchanged. Rows of E are already in
# row_idx order, so we can zip directly.
combined_embedding = dict(zip(index_df["name"].values, E))

print(f"Combined embedding: {len(combined_embedding)} games, dim {E.shape[1]}")


Combined embedding: 26120 games, dim 1584


In [2]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def find_closest_games(target_game, embeddings_dict, top_n=5):
    """
    Finds the most similar games based on their embedding vectors.
    
    :param target_game: Name of the game to search for (str)
    :param embeddings_dict: Dictionary mapping Game Name -> Numpy Array
    :param top_n: Number of recommendations to return
    """
    
    if target_game not in embeddings_dict:
        return f"Error: '{target_game}' not found in the embeddings database."

    target_vector = embeddings_dict[target_game].reshape(1, -1)
    
    game_names = list(embeddings_dict.keys())
    all_vectors = np.array(list(embeddings_dict.values()))
    
    similarity_scores = cosine_similarity(target_vector, all_vectors)[0]
    ranked_indexes = np.argsort(similarity_scores)[::-1]
    
    print(f"Games most similar to '{target_game}':\n")
    results = []
    for idx in ranked_indexes:
        match_name = game_names[idx]
        score = similarity_scores[idx]
        if match_name == target_game:
            continue
        results.append((match_name, score))
        print(f"{len(results)}. {match_name} (Similarity Score: {score:.4f})")
        if len(results) == top_n:
            break
    return results


## Action RPG / open world — *The Witcher 3: Wild Hunt*


In [3]:
find_closest_games("The Witcher 3: Wild Hunt", combined_embedding, top_n=10)


Games most similar to 'The Witcher 3: Wild Hunt':

1. The Elder Scrolls V: Skyrim Special Edition (Similarity Score: 0.6942)
2. The Witcher (Similarity Score: 0.6915)
3. Dark Souls III (Similarity Score: 0.6707)
4. The Witcher 2: Assassins of Kings (Similarity Score: 0.6611)
5. Dragon Age: Inquisition (Similarity Score: 0.6514)
6. Game of Thrones: Kingsroad (Similarity Score: 0.6439)
7. Horizon Zero Dawn: The Frozen Wilds (Similarity Score: 0.6401)
8. Bloodborne: The Old Hunters (Similarity Score: 0.6303)
9. The Elder Scrolls V: Skyrim (Similarity Score: 0.6301)
10. The Witcher 3: Wild Hunt - New Quest: 'Where the Cat and Wolf Play...' (Similarity Score: 0.6299)


[('The Elder Scrolls V: Skyrim Special Edition', np.float32(0.6941887)),
 ('The Witcher', np.float32(0.6914525)),
 ('Dark Souls III', np.float32(0.67074144)),
 ('The Witcher 2: Assassins of Kings', np.float32(0.6610945)),
 ('Dragon Age: Inquisition', np.float32(0.6513698)),
 ('Game of Thrones: Kingsroad', np.float32(0.64392066)),
 ('Horizon Zero Dawn: The Frozen Wilds', np.float32(0.640054)),
 ('Bloodborne: The Old Hunters', np.float32(0.6303459)),
 ('The Elder Scrolls V: Skyrim', np.float32(0.6300989)),
 ("The Witcher 3: Wild Hunt - New Quest: 'Where the Cat and Wolf Play...'",
  np.float32(0.6299416))]

## FPS — *DOOM Eternal*


In [4]:
find_closest_games("DOOM Eternal", combined_embedding, top_n=10)


Games most similar to 'DOOM Eternal':

1. DOOM II (Similarity Score: 0.7544)
2. DOOM: The Dark Ages (Similarity Score: 0.7341)
3. DOOM + DOOM II (Similarity Score: 0.6734)
4. Halo Infinite (Similarity Score: 0.6729)
5. Quake (Similarity Score: 0.6718)
6. DOOM VFR (Similarity Score: 0.6637)
7. Titanfall 2 (Similarity Score: 0.6491)
8. Prodeus (Similarity Score: 0.6452)
9. Call of Duty: Vanguard (Similarity Score: 0.6337)
10. Deathloop (Similarity Score: 0.6332)


[('DOOM II', np.float32(0.7544156)),
 ('DOOM: The Dark Ages', np.float32(0.73413926)),
 ('DOOM + DOOM II', np.float32(0.6733966)),
 ('Halo Infinite', np.float32(0.6729355)),
 ('Quake', np.float32(0.6717634)),
 ('DOOM VFR', np.float32(0.663691)),
 ('Titanfall 2', np.float32(0.64913785)),
 ('Prodeus', np.float32(0.64521664)),
 ('Call of Duty: Vanguard', np.float32(0.63365746)),
 ('Deathloop', np.float32(0.63317204))]

## 4X strategy — *Sid Meier's Civilization VI*


In [5]:
find_closest_games("Sid Meier's Civilization VI", combined_embedding, top_n=10)


Games most similar to 'Sid Meier's Civilization VI':

1. Sid Meier's Civilization V (Similarity Score: 0.8325)
2. Sid Meier's Civilization VII (Similarity Score: 0.8025)
3. Sid Meier's Civilization IV (Similarity Score: 0.7978)
4. Sid Meier's Civilization III (Similarity Score: 0.7592)
5. Sid Meier's Civilization III Complete (Similarity Score: 0.7342)
6. Sid Meier's Civilization IV: Colonization (Similarity Score: 0.7074)
7. Sid Meier's Civilization: Beyond Earth (Similarity Score: 0.6922)
8. Sid Meier's Civilization Revolution 2 (Similarity Score: 0.6771)
9. Total War: WARHAMMER (Similarity Score: 0.6655)
10. Age of Wonders: Planetfall (Similarity Score: 0.6552)


[("Sid Meier's Civilization V", np.float32(0.83245105)),
 ("Sid Meier's Civilization VII", np.float32(0.8024821)),
 ("Sid Meier's Civilization IV", np.float32(0.79775923)),
 ("Sid Meier's Civilization III", np.float32(0.7591691)),
 ("Sid Meier's Civilization III Complete", np.float32(0.7341609)),
 ("Sid Meier's Civilization IV: Colonization", np.float32(0.7073658)),
 ("Sid Meier's Civilization: Beyond Earth", np.float32(0.69224805)),
 ("Sid Meier's Civilization Revolution 2", np.float32(0.6770607)),
 ('Total War: WARHAMMER', np.float32(0.665477)),
 ('Age of Wonders: Planetfall', np.float32(0.6551676))]

## Indie metroidvania — *Hollow Knight*


In [6]:
find_closest_games("Hollow Knight", combined_embedding, top_n=10)


Games most similar to 'Hollow Knight':

1. Hollow Knight: Silksong (Similarity Score: 0.7390)
2. Shovel Knight Dig (Similarity Score: 0.6728)
3. Castle In The Darkness (Similarity Score: 0.6668)
4. The Knight Witch (Similarity Score: 0.6585)
5. Shovel Knight: Treasure Trove (Similarity Score: 0.6474)
6. Celeste (Similarity Score: 0.6459)
7. Blue Fire (Similarity Score: 0.6414)
8. Undertale (Similarity Score: 0.6332)
9. Shovel Knight: King of Cards (Similarity Score: 0.6329)
10. Knight Terrors (Similarity Score: 0.6300)


[('Hollow Knight: Silksong', np.float32(0.7389862)),
 ('Shovel Knight Dig', np.float32(0.67279994)),
 ('Castle In The Darkness', np.float32(0.66679394)),
 ('The Knight Witch', np.float32(0.65852785)),
 ('Shovel Knight: Treasure Trove', np.float32(0.64736056)),
 ('Celeste', np.float32(0.64586765)),
 ('Blue Fire', np.float32(0.64143705)),
 ('Undertale', np.float32(0.63317525)),
 ('Shovel Knight: King of Cards', np.float32(0.63286084)),
 ('Knight Terrors', np.float32(0.6299896))]

## Co-op / puzzle — *Portal 2*


In [7]:
find_closest_games("Portal 2", combined_embedding, top_n=10)


Games most similar to 'Portal 2':

1. Half-Life 2: Episode Two (Similarity Score: 0.6681)
2. Half-Life 2: Episode One (Similarity Score: 0.6664)
3. Team Fortress 2 (Similarity Score: 0.6625)
4. World of Goo (Similarity Score: 0.6496)
5. Half-Life 2 (Similarity Score: 0.6387)
6. Half-Life (Similarity Score: 0.6106)
7. Fallout: New Vegas (Similarity Score: 0.6104)
8. realMyst: Masterpiece Edition (Similarity Score: 0.6052)
9. Crysis Wars (Similarity Score: 0.6051)
10. Quake II (Similarity Score: 0.6018)


[('Half-Life 2: Episode Two', np.float32(0.6681306)),
 ('Half-Life 2: Episode One', np.float32(0.66639936)),
 ('Team Fortress 2', np.float32(0.66254675)),
 ('World of Goo', np.float32(0.6496432)),
 ('Half-Life 2', np.float32(0.63866234)),
 ('Half-Life', np.float32(0.6106024)),
 ('Fallout: New Vegas', np.float32(0.6103878)),
 ('realMyst: Masterpiece Edition', np.float32(0.6051674)),
 ('Crysis Wars', np.float32(0.60505676)),
 ('Quake II', np.float32(0.60177124))]

## Ad-hoc queries

Use this cell to spot-check any game you care about.


In [16]:
find_closest_games("FIFA 15", combined_embedding, top_n=10)


Games most similar to 'FIFA 15':

1. FIFA 14 (Similarity Score: 0.9184)
2. FIFA 17 (Similarity Score: 0.8283)
3. Madden NFL 15 (Similarity Score: 0.8235)
4. FIFA Soccer 13 (Similarity Score: 0.8148)
5. FIFA 18 (Similarity Score: 0.7866)
6. Pro Evolution Soccer 2016 (Similarity Score: 0.7828)
7. Madden NFL 16 (Similarity Score: 0.7744)
8. FIFA World (Similarity Score: 0.7469)
9. FIFA Soccer 12 (Similarity Score: 0.7450)
10. Pro Evolution Soccer 2015 (Similarity Score: 0.7378)


[('FIFA 14', np.float32(0.9183548)),
 ('FIFA 17', np.float32(0.8282548)),
 ('Madden NFL 15', np.float32(0.82353914)),
 ('FIFA Soccer 13', np.float32(0.8148408)),
 ('FIFA 18', np.float32(0.7866425)),
 ('Pro Evolution Soccer 2016', np.float32(0.782751)),
 ('Madden NFL 16', np.float32(0.77435696)),
 ('FIFA World', np.float32(0.74694765)),
 ('FIFA Soccer 12', np.float32(0.7450113)),
 ('Pro Evolution Soccer 2015', np.float32(0.73778075))]

## Combined vs tags-only — does the concat add signal?

Side-by-side comparison: top-10 from the combined embedding (title + description + tags + collab + scalars) against top-10 from the standalone tags embedding (genres + keywords only). Differences show what the title/description/collab blocks pull in on top of pure genre overlap.


In [9]:
with open("../../data/game_tags_embeddings.pkl", "rb") as f:
    tags_only_embedding = pickle.load(f)

import io, contextlib

def compare_embeddings(target_game, top_n=10):
    """Runs find_closest_games on both embedding dicts and prints results side by side."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        results_combined = find_closest_games(target_game, combined_embedding, top_n=top_n)
        results_tags     = find_closest_games(target_game, tags_only_embedding, top_n=top_n)

    if isinstance(results_combined, str) or isinstance(results_tags, str):
        print(results_combined if isinstance(results_combined, str) else results_tags)
        return

    max_len = max(len(results_combined), len(results_tags))
    results_combined = list(results_combined) + [("—", float("nan"))] * (max_len - len(results_combined))
    results_tags     = list(results_tags)     + [("—", float("nan"))] * (max_len - len(results_tags))

    print(f"{'=' * 90}")
    print(f"  Comparing recommendations for: '{target_game}'")
    print(f"{'=' * 90}")
    print(f"{'Rank':<6} {'combined (1584-d)':<44} {'tags only (32-d)'}")
    print(f"{'-' * 90}")
    for rank, ((name_c, score_c), (name_t, score_t)) in enumerate(zip(results_combined, results_tags), 1):
        score_c_str = f"{score_c:.4f}" if score_c == score_c else "—"
        score_t_str = f"{score_t:.4f}" if score_t == score_t else "—"
        col_c = f"{name_c} ({score_c_str})"
        col_t = f"{name_t} ({score_t_str})"
        print(f"{rank:<6} {col_c:<44} {col_t}")

compare_embeddings("The Witcher 3: Wild Hunt", top_n=10)


  Comparing recommendations for: 'The Witcher 3: Wild Hunt'
Rank   combined (1584-d)                            tags only (32-d)
------------------------------------------------------------------------------------------
1      The Elder Scrolls V: Skyrim Special Edition (0.6942) Jade Empire: Special Edition (0.9616)
2      The Witcher (0.6915)                         Fable: The Lost Chapters (0.9614)
3      Dark Souls III (0.6707)                      Dark Souls: Prepare To Die Edition (0.9488)
4      The Witcher 2: Assassins of Kings (0.6611)   Risen 2: Dark Waters (0.9449)
5      Dragon Age: Inquisition (0.6514)             Risen (0.9358)
6      Game of Thrones: Kingsroad (0.6439)          Risen 3: Titan Lords (0.9279)
7      Horizon Zero Dawn: The Frozen Wilds (0.6401) Horizon Zero Dawn Complete Edition (0.9152)
8      Bloodborne: The Old Hunters (0.6303)         Lies of P (0.9148)
9      The Elder Scrolls V: Skyrim (0.6301)         Mass Effect 3 (0.9122)
10     The Witcher 3: Wild 

In [10]:
compare_embeddings("DOOM Eternal", top_n=10)


  Comparing recommendations for: 'DOOM Eternal'
Rank   combined (1584-d)                            tags only (32-d)
------------------------------------------------------------------------------------------
1      DOOM II (0.7544)                             DOOM II (0.9591)
2      DOOM: The Dark Ages (0.7341)                 Wrath: Aeon of Ruin (0.9557)
3      DOOM + DOOM II (0.6734)                      Blood: Fresh Supply (0.9415)
4      Halo Infinite (0.6729)                       F.E.A.R. 2: Project Origin (0.9391)
5      Quake (0.6718)                               Kingpin: Life of Crime (0.9248)
6      DOOM VFR (0.6637)                            Wolfenstein II: The New Colossus (0.9063)
7      Titanfall 2 (0.6491)                         Quake 4 (0.9048)
8      Prodeus (0.6452)                             Gynophobia (0.9042)
9      Call of Duty: Vanguard (0.6337)              Return to Castle Wolfenstein (0.9038)
10     Deathloop (0.6332)                           Serious Sam: